# GNN Retrieval - 3-Phase Training Plan (`Chunk`-`Entity`)

This notebook is dedicated to the multi-seed, multi-phase training pipeline and phase-by-phase comparison.


In [1]:
# При необходимости раскомментируйте установку зависимостей
%pip install neo4j pandas numpy scikit-learn torch torch-geometric qdrant-client



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [46]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from neo4j import GraphDatabase
from qdrant_client import QdrantClient
from qdrant_client.models import FieldCondition, Filter, MatchAny
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


device(type='cuda')

In [55]:
# Конфиг подключения
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

QDRANT_URL = os.getenv('QDRANT_URL', 'http://localhost:6333')
QDRANT_API_KEY = os.getenv('QDRANT_API_KEY')
QDRANT_CHUNK_COLLECTION = os.getenv('QDRANT_CHUNK_COLLECTION', 'legal_chunks_vectors')
QDRANT_ENTITY_COLLECTION = os.getenv('QDRANT_ENTITY_COLLECTION', 'legal_entities_vectors')
QDRANT_ID_FIELD = os.getenv('QDRANT_ID_FIELD', 'id')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

def run_query(query: str, params: dict | None = None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [r.data() for r in result]

schema_probe = run_query("""
MATCH (c:Chunk)
OPTIONAL MATCH (c)-[:MENTIONS]->(e:Entity)
RETURN count(DISTINCT c) AS chunks, count(DISTINCT e) AS entities, count(*) AS mention_rows
""")
schema_probe


[{'chunks': 821, 'entities': 2895, 'mention_rows': 4745}]

In [56]:
# Вытягиваем узлы/ребра и embedding_id для загрузки векторов из Qdrant
chunks = run_query("""
MATCH (c:Chunk)
RETURN
  c.chunk_id AS chunk_id,
  c.doc_id AS doc_id,
  c.embedding_id AS embedding_id,
  coalesce(c.text, '') AS text
""")

entities = run_query("""
MATCH (e:Entity)
RETURN
  e.entity_id AS entity_id,
  coalesce(e.embedding_id, e.entity_id) AS embedding_id,
  coalesce(e.entity_name, e.normalized_value, e.value, '') AS entity_text
""")

mentions = run_query("""
MATCH (c:Chunk)-[:MENTIONS]->(e:Entity)
RETURN c.chunk_id AS chunk_id, e.entity_id AS entity_id
""")

related = run_query("""
MATCH (e1:Entity)-[:RELATED_TO]->(e2:Entity)
RETURN e1.entity_id AS src_entity_id, e2.entity_id AS dst_entity_id
""")

chunks_df = pd.DataFrame(chunks).dropna(subset=['chunk_id', 'embedding_id']).drop_duplicates('chunk_id')
entities_df = pd.DataFrame(entities).dropna(subset=['entity_id', 'embedding_id']).drop_duplicates('entity_id')
mentions_df = pd.DataFrame(mentions).dropna().drop_duplicates()
related_df = pd.DataFrame(related).dropna().drop_duplicates()

print('chunks (with embedding_id):', len(chunks_df))
print('entities (with embedding_id):', len(entities_df))
print('mentions:', len(mentions_df))
print('related_to:', len(related_df))
chunks_df.head(2), entities_df.head(2), mentions_df.head(2)


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `text` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=7, column=14, offset=119>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 7, 'column': 14}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (c:Chunk)\nRETURN\n  c.chunk_id AS chunk_id,\n  c.doc_id AS doc_id,\n  c.embedding_id AS embedding_id,\n  coalesce(c.text, '') AS text\n"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `normalized_value` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=6, column=29, offset=138>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 138, 'line': 6, 'column': 29}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (e:Entity)\nRETURN\n  e.entity_id AS entity_id,\n  coalesce(e.embedding_id, e.entity_id) AS embedding_id,\n  coalesce(e.entity_name, e.normalized_value, e.value, '') AS entity_text\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. 

chunks (with embedding_id): 821
entities (with embedding_id): 2895
mentions: 4745
related_to: 3048


(                               chunk_id doc_id  \
 0  019d8288-d132-71ed-b343-9c65524dcb53     12   
 1  019d8288-d136-7bfd-accb-06a906af2fe0     12   
 
                            embedding_id text  
 0  b22e00fa-af96-5b15-8d49-4f770d79cc21       
 1  d02ca84d-1bca-5a0f-8ec0-08810d645b90       ,
                                            entity_id  \
 0  f7381fc918f9933102106f5799248780039f5987967067...   
 1  8f4e8b62535fb495bb91ff066c856652416cdd676a8615...   
 
                            embedding_id        entity_text  
 0  68363780-c85c-5712-be9c-d811cab98174      Paris Commune  
 1  124ebd02-c765-5073-82f5-174cc956e4ed  Russian Civil War  ,
                                chunk_id  \
 0  019d8288-d132-71ed-b343-9c65524dcb53   
 1  019d8288-d132-71ed-b343-9c65524dcb53   
 
                                            entity_id  
 0  2246cc1e9afc11f58ec447ae8eb7cd7f6091f5b4672428...  
 1  15b38175c1f72394880594a9d93b640e5d14c9db0029ae...  )

In [57]:
# Загружаем векторы из Qdrant по embedding_id (= point.id в Qdrant)
def _extract_vector(raw_vector):
    if raw_vector is None:
        return None
    if isinstance(raw_vector, dict):
        # named vectors: берём первый
        raw_vector = next(iter(raw_vector.values())) if raw_vector else None
    if raw_vector is None:
        return None
    return np.asarray(raw_vector, dtype=np.float32)


def _normalize_qdrant_point_id(raw_id):
    if raw_id is None or pd.isna(raw_id):
        return None
    if isinstance(raw_id, (int, np.integer)):
        return int(raw_id)

    raw_id = str(raw_id).strip()
    if not raw_id:
        return None
    if raw_id.lstrip('-').isdigit():
        return int(raw_id)
    return raw_id


def fetch_vectors_by_embedding_ids(collection_name: str, embedding_ids: list[str], batch_size: int = 256):
    vectors: dict[str, np.ndarray] = {}
    ids = []
    for raw_id in embedding_ids:
        normalized_id = _normalize_qdrant_point_id(raw_id)
        if normalized_id is not None:
            ids.append(normalized_id)

    ids = list(dict.fromkeys(ids))
    for i in range(0, len(ids), batch_size):
        batch = ids[i:i+batch_size]
        points = qdrant.retrieve(
            collection_name=collection_name,
            ids=batch,
            with_payload=False,
            with_vectors=True,
        )
        for p in points:
            v = _extract_vector(p.vector)
            if v is not None:
                vectors[str(p.id)] = v
    return vectors


chunk_vec_map = fetch_vectors_by_embedding_ids(
    collection_name=QDRANT_CHUNK_COLLECTION,
    embedding_ids=chunks_df['embedding_id'].tolist(),
)
entity_vec_map = fetch_vectors_by_embedding_ids(
    collection_name=QDRANT_ENTITY_COLLECTION,
    embedding_ids=entities_df['embedding_id'].tolist(),
)

chunks_df['vec'] = chunks_df['embedding_id'].astype(str).map(chunk_vec_map)
entities_df['vec'] = entities_df['embedding_id'].astype(str).map(entity_vec_map)

chunks_df = chunks_df[chunks_df['vec'].notna()].copy().reset_index(drop=True)
entities_df = entities_df[entities_df['vec'].notna()].copy().reset_index(drop=True)

#print(entities_df)
print("ENTITIES COUNT: ", entities_df.count())

if chunks_df.empty or entities_df.empty:
    raise ValueError('Не удалось получить вектора из Qdrant. Проверьте collection names и embedding_id mapping.')

chunk_dims = chunks_df['vec'].apply(len).unique().tolist()
entity_dims = entities_df['vec'].apply(len).unique().tolist()
if len(chunk_dims) != 1 or len(entity_dims) != 1:
    raise ValueError(f'Непостоянная размерность векторов: chunk={chunk_dims}, entity={entity_dims}')

chunk_x = np.vstack(chunks_df['vec'].values).astype(np.float32)
entity_x = np.vstack(entities_df['vec'].values).astype(np.float32)

# L2-нормализация входных фичей
chunk_x = chunk_x / (np.linalg.norm(chunk_x, axis=1, keepdims=True) + 1e-12)
entity_x = entity_x / (np.linalg.norm(entity_x, axis=1, keepdims=True) + 1e-12)

chunk2idx = {cid: i for i, cid in enumerate(chunks_df['chunk_id'].tolist())}
entity2idx = {eid: i for i, eid in enumerate(entities_df['entity_id'].tolist())}

mentions_df = mentions_df[mentions_df['chunk_id'].isin(chunk2idx) & mentions_df['entity_id'].isin(entity2idx)].copy()
related_df = related_df[related_df['src_entity_id'].isin(entity2idx) & related_df['dst_entity_id'].isin(entity2idx)].copy()

mention_src = torch.tensor([chunk2idx[x] for x in mentions_df['chunk_id']], dtype=torch.long)
mention_dst = torch.tensor([entity2idx[x] for x in mentions_df['entity_id']], dtype=torch.long)

rel_src = torch.tensor([entity2idx[x] for x in related_df['src_entity_id']], dtype=torch.long) if len(related_df) else torch.empty(0, dtype=torch.long)
rel_dst = torch.tensor([entity2idx[x] for x in related_df['dst_entity_id']], dtype=torch.long) if len(related_df) else torch.empty(0, dtype=torch.long)

data = HeteroData()
data['chunk'].x = torch.from_numpy(chunk_x)
data['entity'].x = torch.from_numpy(entity_x)

data['chunk', 'mentions', 'entity'].edge_index = torch.stack([mention_src, mention_dst], dim=0)
data['entity', 'mentioned_in', 'chunk'].edge_index = torch.stack([mention_dst, mention_src], dim=0)
data['entity', 'related_to', 'entity'].edge_index = torch.stack([rel_src, rel_dst], dim=0)
data['entity', 'related_to_rev', 'entity'].edge_index = torch.stack([rel_dst, rel_src], dim=0)

print(f'Loaded Qdrant vectors: chunks={len(chunks_df)} (dim={chunk_x.shape[1]}), entities={len(entities_df)} (dim={entity_x.shape[1]})')
data


ENTITIES COUNT:  entity_id       2895
embedding_id    2895
entity_text     2895
vec             2895
dtype: int64
Loaded Qdrant vectors: chunks=821 (dim=256), entities=2895 (dim=256)


HeteroData(
  chunk={ x=[821, 256] },
  entity={ x=[2895, 256] },
  (chunk, mentions, entity)={ edge_index=[2, 4745] },
  (entity, mentioned_in, chunk)={ edge_index=[2, 4745] },
  (entity, related_to, entity)={ edge_index=[2, 3048] },
  (entity, related_to_rev, entity)={ edge_index=[2, 3048] }
)

In [58]:
# Train/Val/Test split по рёбрам MENTIONS
all_edges = data['chunk', 'mentions', 'entity'].edge_index.t().cpu().numpy()
perm = np.random.permutation(len(all_edges))
all_edges = all_edges[perm]

n = len(all_edges)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_edges = torch.tensor(all_edges[:n_train], dtype=torch.long)
val_edges = torch.tensor(all_edges[n_train:n_train+n_val], dtype=torch.long)
test_edges = torch.tensor(all_edges[n_train+n_val:], dtype=torch.long)

# Для честной валидации/теста убираем val/test MENTIONS-рёбра из графа обучения.
train_edge_index = train_edges.t().contiguous()
train_data = data.clone()
train_data['chunk', 'mentions', 'entity'].edge_index = train_edge_index
train_data['entity', 'mentioned_in', 'chunk'].edge_index = train_edge_index.flip(0)

num_chunks = train_data['chunk'].x.shape[0]
num_entities = train_data['entity'].x.shape[0]

train_positives_by_chunk: dict[int, set[int]] = {}
for c_idx, e_idx in train_edges.tolist():
    train_positives_by_chunk.setdefault(int(c_idx), set()).add(int(e_idx))

all_positives_by_chunk: dict[int, set[int]] = {}
for c_idx, e_idx in all_edges.tolist():
    all_positives_by_chunk.setdefault(int(c_idx), set()).add(int(e_idx))


def build_hard_negative_pools(
    chunk_features: np.ndarray,
    entity_features: np.ndarray,
    positives_map: dict[int, set[int]],
    top_k: int = 64,
    batch_size: int = 128,
):
    chunk_tensor = torch.from_numpy(chunk_features)
    entity_tensor = torch.from_numpy(entity_features)
    pools: dict[int, list[int]] = {}

    for start in range(0, len(chunk_tensor), batch_size):
        stop = min(start + batch_size, len(chunk_tensor))
        scores = torch.matmul(chunk_tensor[start:stop], entity_tensor.t())
        candidate_count = min(entity_tensor.shape[0], top_k + 96)
        top_idx = torch.topk(scores, k=candidate_count, dim=1).indices.cpu().tolist()

        for local_idx, candidates in enumerate(top_idx):
            chunk_idx = start + local_idx
            positives = positives_map.get(chunk_idx, set())
            hard_candidates = []
            for entity_idx in candidates:
                entity_idx = int(entity_idx)
                if entity_idx in positives:
                    continue
                hard_candidates.append(entity_idx)
                if len(hard_candidates) == top_k:
                    break
            pools[chunk_idx] = hard_candidates

    return pools


hard_negative_top_k = 64
hard_negative_pools = build_hard_negative_pools(
    chunk_features=chunk_x,
    entity_features=entity_x,
    positives_map=train_positives_by_chunk,
    top_k=hard_negative_top_k,
)


def sample_negative_entities(
    batch_edges: torch.Tensor,
    positives_map: dict[int, set[int]],
    num_entities: int,
    hard_pools: dict[int, list[int]] | None = None,
    num_random_neg: int = 4,
    num_hard_neg: int = 2,
    max_tries: int = 50,
) -> torch.Tensor:
    total_neg = num_random_neg + num_hard_neg
    neg_entity_ids = torch.empty((len(batch_edges), total_neg), dtype=torch.long)

    for i, (c_idx, e_idx) in enumerate(batch_edges.tolist()):
        c_idx = int(c_idx)
        e_idx = int(e_idx)
        positives = positives_map.get(c_idx, set())
        selected = set()

        candidate_hards = [
            int(entity_idx)
            for entity_idx in (hard_pools or {}).get(c_idx, [])
            if int(entity_idx) != e_idx and int(entity_idx) not in positives
        ]
        hard_take = min(num_hard_neg, len(candidate_hards))
        if hard_take:
            for hard_entity in random.sample(candidate_hards, k=hard_take):
                selected.add(hard_entity)

        while len(selected) < total_neg:
            found = False
            for _ in range(max_tries):
                candidate = random.randrange(num_entities)
                if candidate != e_idx and candidate not in positives and candidate not in selected:
                    selected.add(candidate)
                    found = True
                    break
            if not found:
                break

        if len(selected) < total_neg:
            raise RuntimeError(f'Не удалось набрать {total_neg} negatives для chunk={c_idx}')

        neg_entity_ids[i] = torch.tensor(sorted(selected), dtype=torch.long)

    return neg_entity_ids

print('Hard-negative pools prepared for chunks:', len(hard_negative_pools))
len(train_edges), len(val_edges), len(test_edges)


Hard-negative pools prepared for chunks: 821


(3796, 474, 475)

In [59]:
class RetrievalHeteroSAGE(nn.Module):
    def __init__(
        self,
        hidden_dim: int = 256,
        out_dim: int = 256,
        num_layers: int = 2,
        dropout: float = 0.10,
        residual_weight: float = 0.85,
        learnable_residual: bool = True,
    ):
        super().__init__()
        if num_layers < 1:
            raise ValueError('num_layers must be >= 1')

        layer_dims = [hidden_dim] * max(num_layers - 1, 0) + [out_dim]
        self.layers = nn.ModuleList([
            HeteroConv({
                ('chunk', 'mentions', 'entity'): SAGEConv((-1, -1), layer_dim),
                ('entity', 'mentioned_in', 'chunk'): SAGEConv((-1, -1), layer_dim),
                ('entity', 'related_to', 'entity'): SAGEConv((-1, -1), layer_dim),
                ('entity', 'related_to_rev', 'entity'): SAGEConv((-1, -1), layer_dim),
            }, aggr='mean')
            for layer_dim in layer_dims
        ])
        self.dropout = dropout
        residual_logit = float(np.log(residual_weight / (1.0 - residual_weight)))
        if learnable_residual:
            self.chunk_residual_logit = nn.Parameter(torch.tensor(residual_logit, dtype=torch.float32))
            self.entity_residual_logit = nn.Parameter(torch.tensor(residual_logit, dtype=torch.float32))
        else:
            self.register_buffer('chunk_residual_logit', torch.tensor(residual_logit, dtype=torch.float32))
            self.register_buffer('entity_residual_logit', torch.tensor(residual_logit, dtype=torch.float32))

    def encode_graph(self, x_dict, edge_index_dict):
        h = {k: F.normalize(v, p=2, dim=-1) for k, v in x_dict.items()}
        for layer_idx, conv in enumerate(self.layers):
            h = conv(h, edge_index_dict)
            if layer_idx != len(self.layers) - 1:
                h = {
                    k: F.dropout(F.relu(v), p=self.dropout, training=self.training)
                    for k, v in h.items()
                }
        return {k: F.normalize(v, p=2, dim=-1) for k, v in h.items()}

    def residual_weights(self):
        return {
            'chunk': float(torch.sigmoid(self.chunk_residual_logit).detach().cpu()),
            'entity': float(torch.sigmoid(self.entity_residual_logit).detach().cpu()),
        }

    def forward(self, x_dict, edge_index_dict):
        raw = {k: F.normalize(v, p=2, dim=-1) for k, v in x_dict.items()}
        graph = self.encode_graph(raw, edge_index_dict)

        chunk_alpha = torch.sigmoid(self.chunk_residual_logit)
        entity_alpha = torch.sigmoid(self.entity_residual_logit)
        fused = {
            'chunk': F.normalize(chunk_alpha * raw['chunk'] + (1.0 - chunk_alpha) * graph['chunk'], p=2, dim=-1),
            'entity': F.normalize(entity_alpha * raw['entity'] + (1.0 - entity_alpha) * graph['entity'], p=2, dim=-1),
        }
        return fused


def edge_scores(chunk_emb: torch.Tensor, entity_emb: torch.Tensor, edges: torch.Tensor) -> torch.Tensor:
    c = chunk_emb[edges[:, 0]]
    e = entity_emb[edges[:, 1]]
    return (c * e).sum(dim=-1)


@torch.no_grad()
def collect_edge_ranks(chunk_emb: torch.Tensor, entity_emb: torch.Tensor, eval_edges: torch.Tensor) -> torch.Tensor:
    ranks = []
    for c_idx, e_idx in eval_edges.tolist():
        scores = torch.matmul(entity_emb, chunk_emb[c_idx])
        rank_pos = (torch.argsort(scores, descending=True) == e_idx).nonzero(as_tuple=False)
        if len(rank_pos):
            ranks.append(int(rank_pos[0].item()) + 1)
    return torch.tensor(ranks, dtype=torch.float32)


@torch.no_grad()
def evaluate_embeddings(chunk_emb: torch.Tensor, entity_emb: torch.Tensor, eval_edges: torch.Tensor, k_list=(5, 10, 20)):
    ranks = collect_edge_ranks(chunk_emb, entity_emb, eval_edges)
    metrics = {}
    for k in k_list:
        metrics[f'Recall@{k}'] = float((ranks <= k).float().mean().item()) if len(ranks) else 0.0
    metrics['MRR'] = float((1.0 / ranks).mean().item()) if len(ranks) else 0.0
    return metrics


@torch.no_grad()
def evaluate_retrieval(model: nn.Module, graph_data: HeteroData, eval_edges: torch.Tensor, k_list=(5, 10, 20)):
    model.eval()
    z = model(graph_data.x_dict, graph_data.edge_index_dict)
    return evaluate_embeddings(z['chunk'], z['entity'], eval_edges, k_list=k_list)


def build_metric_table(split_name: str, experiment_name: str, metrics: dict[str, float], baseline_metrics: dict[str, float]):
    rows = []
    for metric_name, metric_value in metrics.items():
        baseline_value = baseline_metrics[metric_name]
        delta_abs = metric_value - baseline_value
        delta_pct = (delta_abs / baseline_value * 100.0) if baseline_value else np.nan
        rows.append({
            'split': split_name,
            'experiment': experiment_name,
            'metric': metric_name,
            'baseline': baseline_value,
            'value': metric_value,
            'delta_abs': delta_abs,
            'delta_pct': delta_pct,
        })
    return rows


@torch.no_grad()
def embedding_diagnostics(
    chunk_emb: torch.Tensor,
    entity_emb: torch.Tensor,
    eval_edges: torch.Tensor,
    positives_map: dict[int, set[int]],
    sample_size: int = 192,
    negatives_per_edge: int = 6,
):
    eval_edges_cpu = eval_edges.detach().cpu()
    if len(eval_edges_cpu) == 0:
        return {}

    if len(eval_edges_cpu) > sample_size:
        sample_idx = torch.randperm(len(eval_edges_cpu))[:sample_size]
        sampled_edges_cpu = eval_edges_cpu[sample_idx]
    else:
        sampled_edges_cpu = eval_edges_cpu

    sampled_edges = sampled_edges_cpu.to(chunk_emb.device)
    pos_scores = edge_scores(chunk_emb, entity_emb, sampled_edges).detach().cpu()
    neg_ids = sample_negative_entities(
        sampled_edges_cpu,
        positives_map=positives_map,
        num_entities=entity_emb.shape[0],
        hard_pools=None,
        num_random_neg=negatives_per_edge,
        num_hard_neg=0,
    ).to(entity_emb.device)
    neg_scores = (entity_emb[neg_ids] * chunk_emb[sampled_edges[:, 0]].unsqueeze(1)).sum(dim=-1).detach().cpu().reshape(-1)
    ranks = collect_edge_ranks(chunk_emb, entity_emb, sampled_edges).detach().cpu()

    return {
        'pos_mean': float(pos_scores.mean().item()),
        'pos_std': float(pos_scores.std(unbiased=False).item()),
        'neg_mean': float(neg_scores.mean().item()),
        'neg_std': float(neg_scores.std(unbiased=False).item()),
        'score_gap': float(pos_scores.mean().item() - neg_scores.mean().item()),
        'median_rank': float(ranks.median().item()) if len(ranks) else np.nan,
        'p90_rank': float(torch.quantile(ranks, 0.90).item()) if len(ranks) else np.nan,
    }


In [60]:
baseline_chunk_emb = torch.from_numpy(chunk_x).to(DEVICE)
baseline_entity_emb = torch.from_numpy(entity_x).to(DEVICE)
baseline_val_metrics = evaluate_embeddings(baseline_chunk_emb, baseline_entity_emb, val_edges)
baseline_test_metrics = evaluate_embeddings(baseline_chunk_emb, baseline_entity_emb, test_edges)

train_data = train_data.to(DEVICE)
train_edges = train_edges.to(DEVICE)
val_edges = val_edges.to(DEVICE)
test_edges = test_edges.to(DEVICE)


# Legacy single-run/ablation block removed.
# The notebook now uses the 3-phase multi-seed pipeline in the next cell.



=== Running best_from_phase1_or2_plus_cosine_clip (seed=52) ===
[best_from_phase1_or2_plus_cosine_clip|seed=52] epoch=001 | loss=2.7828 | ranking_loss=2.7824 | preserve_loss=0.0181 | val={'Recall@5': 0.12447257339954376, 'Recall@10': 0.18987341225147247, 'Recall@20': 0.2805907130241394, 'MRR': 0.08598919212818146} | patience_left=12 | mined_chunks=817
[best_from_phase1_or2_plus_cosine_clip|seed=52] epoch=010 | loss=2.7676 | ranking_loss=2.7672 | preserve_loss=0.0182 | val={'Recall@5': 0.13924050331115723, 'Recall@10': 0.2067510485649109, 'Recall@20': 0.31012657284736633, 'MRR': 0.09463363885879517} | patience_left=12
[best_from_phase1_or2_plus_cosine_clip|seed=52] epoch=020 | loss=2.7530 | ranking_loss=2.7526 | preserve_loss=0.0178 | val={'Recall@5': 0.15189872682094574, 'Recall@10': 0.2194092869758606, 'Recall@20': 0.31856539845466614, 'MRR': 0.10122793167829514} | patience_left=12
[best_from_phase1_or2_plus_cosine_clip|seed=52] epoch=030 | loss=2.7435 | ranking_loss=2.7431 | preserv

In [53]:
# Comparison is consolidated in the 3-phase runner cell below (cell 10).
# This placeholder keeps execution order simple when running the whole notebook.
print('Run cell 10 for phase-by-phase comparison and final model selection.')


Baseline val metrics: {'Recall@5': 0.1463414579629898, 'Recall@10': 0.21507760882377625, 'Recall@20': 0.3082039952278137, 'MRR': 0.09335071593523026}
Selected GNN val metrics: {'Recall@5': 0.18847006559371948, 'Recall@10': 0.26829269528388977, 'Recall@20': 0.3569844663143158, 'MRR': 0.1327439397573471}
Baseline test metrics: {'Recall@5': 0.14159291982650757, 'Recall@10': 0.20353981852531433, 'Recall@20': 0.3097345232963562, 'MRR': 0.09821619838476181}
Selected GNN test metrics: {'Recall@5': 0.1703539788722992, 'Recall@10': 0.25221237540245056, 'Recall@20': 0.34292036294937134, 'MRR': 0.11655783653259277}

Metric deltas vs baseline (% and absolute):
split                            experiment    metric  baseline  value  delta_abs  delta_pct
  val best_from_phase1_or2_plus_cosine_clip  Recall@5    0.1463 0.1885     0.0421    28.7879
  val best_from_phase1_or2_plus_cosine_clip Recall@10    0.2151 0.2683     0.0532    24.7423
  val best_from_phase1_or2_plus_cosine_clip Recall@20    0.3082 

,experiment,val_MRR,test_MRR,val_Recall@10,test_Recall@10,chunk_alpha,entity_alpha
0,best_from_phase1_or2_plus_cosine_clip,0.132744,0.116558,0.268293,0.252212,0.878659,0.878537


In [ ]:
# Plan runner: 12 experiments x 3 seeds with phase gates
from collections import defaultdict
from torch.nn.parameter import UninitializedParameter

SEEDS = [42, 52, 62]
MAX_EPOCHS = 90
PATIENCE = 12
VAL_IMPROVEMENT_THRESHOLD = 0.005
MAX_ALLOWED_TEST_DROP = 0.002
STD_THRESHOLD = 0.01

baseline_val_diag = embedding_diagnostics(
    baseline_chunk_emb,
    baseline_entity_emb,
    val_edges,
    positives_map=all_positives_by_chunk,
)
baseline_test_diag = embedding_diagnostics(
    baseline_chunk_emb,
    baseline_entity_emb,
    test_edges,
    positives_map=all_positives_by_chunk,
)


def reset_seeds_with_value(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class EMAHelper:
    def __init__(self, model: nn.Module, decay: float = 0.995):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        self._init_from_model(model)

    @staticmethod
    def _is_ready_param(param: nn.Parameter) -> bool:
        return not isinstance(param, UninitializedParameter)

    @torch.no_grad()
    def _init_from_model(self, model: nn.Module):
        for name, param in model.named_parameters():
            if not param.requires_grad or not self._is_ready_param(param):
                continue
            if name not in self.shadow:
                self.shadow[name] = param.detach().clone()

    @torch.no_grad()
    def update(self, model: nn.Module):
        self._init_from_model(model)
        for name, param in model.named_parameters():
            if not param.requires_grad or not self._is_ready_param(param):
                continue
            if name not in self.shadow:
                self.shadow[name] = param.detach().clone()
            else:
                self.shadow[name].mul_(self.decay).add_(param.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def apply(self, model: nn.Module):
        self._init_from_model(model)
        self.backup = {}
        for name, param in model.named_parameters():
            if not param.requires_grad or not self._is_ready_param(param):
                continue
            if name not in self.shadow:
                continue
            self.backup[name] = param.detach().clone()
            param.data.copy_(self.shadow[name].data)

    @torch.no_grad()
    def restore(self, model: nn.Module):
        for name, param in model.named_parameters():
            if not param.requires_grad or not self._is_ready_param(param):
                continue
            if name in self.backup:
                param.data.copy_(self.backup[name].data)
        self.backup = {}


@torch.no_grad()
def _clone_state_dict(model: nn.Module):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def train_experiment_seeded(config: dict, seed: int, control_val_mrr: float | None = None):
    reset_seeds_with_value(seed)

    model = RetrievalHeteroSAGE(
        hidden_dim=config.get('hidden_dim', 256),
        out_dim=config.get('out_dim', 256),
        num_layers=config.get('num_layers', 1),
        dropout=config.get('dropout', 0.05),
        residual_weight=config.get('residual_weight', 0.88),
        learnable_residual=config.get('learnable_residual', True),
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.get('lr', 3e-4),
        weight_decay=config.get('weight_decay', 1e-5),
    )

    scheduler = None
    if config.get('scheduler') == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=config.get('max_epochs', MAX_EPOCHS),
            eta_min=config.get('min_lr', 1e-5),
        )
    elif config.get('scheduler') == 'one_cycle':
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=config.get('max_lr', config.get('lr', 3e-4) * 3.0),
            epochs=config.get('max_epochs', MAX_EPOCHS),
            steps_per_epoch=1,
            pct_start=0.3,
        )

    use_ema = bool(config.get('use_ema', False))
    ema = EMAHelper(model, decay=config.get('ema_decay', 0.995)) if use_ema else None

    max_epochs = config.get('max_epochs', MAX_EPOCHS)
    patience = config.get('patience', PATIENCE)
    min_epochs = config.get('min_epochs', 12)
    patience_left = patience

    best_val_mrr = float('-inf')
    best_state = None
    best_epoch = 0

    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()

        z = model(train_data.x_dict, train_data.edge_index_dict)
        chunk_batch = z['chunk'][train_edges[:, 0]]
        pos_entity_batch = z['entity'][train_edges[:, 1]]

        hard_pools = hard_negative_pools
        num_random = config.get('num_random_neg', 4)
        num_hard = config.get('num_hard_neg', 2)

        # Curriculum for hard negatives.
        if config.get('curriculum_neg', False):
            progress = epoch / max_epochs
            if progress < 0.34:
                num_hard = 2
                num_random = max(1, num_random + config.get('num_hard_neg', 6) - 2)
            elif progress < 0.67:
                num_hard = 4
                num_random = max(1, num_random + config.get('num_hard_neg', 6) - 4)
            else:
                num_hard = config.get('num_hard_neg', 6)
                num_random = config.get('num_random_neg', 2)

        neg_entity_ids = sample_negative_entities(
            train_edges.detach().cpu(),
            positives_map=train_positives_by_chunk,
            num_entities=num_entities,
            hard_pools=hard_pools,
            num_random_neg=num_random,
            num_hard_neg=num_hard,
        ).to(DEVICE)

        neg_scores = (z['entity'][neg_entity_ids] * chunk_batch.unsqueeze(1)).sum(dim=-1)
        pos_scores = (chunk_batch * pos_entity_batch).sum(dim=-1)

        loss_name = config.get('loss_name', 'cross_entropy')
        if loss_name == 'cross_entropy':
            logits = torch.cat([pos_scores.unsqueeze(1), neg_scores], dim=1)
            labels = torch.zeros(logits.size(0), dtype=torch.long, device=DEVICE)
            ranking_loss = F.cross_entropy(logits, labels)
        elif loss_name == 'pairwise_softplus':
            margin = config.get('margin', 0.10)
            ranking_loss = F.softplus(neg_scores - pos_scores.unsqueeze(1) + margin).mean()
        else:
            raise ValueError(f'Unsupported loss_name: {loss_name}')

        preserve_loss = 0.5 * (
            (chunk_batch - raw_chunk_train[train_edges[:, 0]]).pow(2).sum(dim=-1).mean()
            + (pos_entity_batch - raw_entity_train[train_edges[:, 1]]).pow(2).sum(dim=-1).mean()
        )
        loss = ranking_loss + config.get('preserve_weight', 0.02) * preserve_loss

        loss.backward()
        grad_clip = config.get('grad_clip')
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        if ema is not None:
            ema.update(model)

        eval_model = model
        if ema is not None:
            ema.apply(model)
            eval_model = model

        val_metrics = evaluate_retrieval(eval_model, train_data, val_edges)
        val_mrr = val_metrics['MRR']

        if ema is not None:
            ema.restore(model)

        history.append({
            'epoch': epoch,
            'loss': float(loss.item()),
            'ranking_loss': float(ranking_loss.item()),
            'preserve_loss': float(preserve_loss.item()),
            'val_mrr': float(val_mrr),
            'val_recall10': float(val_metrics['Recall@10']),
        })

        if val_mrr > best_val_mrr:
            best_val_mrr = val_mrr
            best_state = _clone_state_dict(model)
            best_epoch = epoch
            patience_left = patience
        else:
            patience_left -= 1

        # Early kill if clearly underperforming control after warmup.
        if (
            control_val_mrr is not None
            and epoch >= max(min_epochs, int(0.4 * max_epochs))
            and best_val_mrr < (control_val_mrr - 0.01)
        ):
            print(f"[{config['name']}|seed={seed}] early-abort at epoch {epoch:03d} (lagging control)")
            break

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"[{config['name']}|seed={seed}] epoch={epoch:03d} "
                f"loss={loss.item():.4f} val_mrr={val_mrr:.6f} "
                f"val_r10={val_metrics['Recall@10']:.4f} patience_left={patience_left}"
            )

        if patience_left <= 0 and epoch >= min_epochs:
            print(
                f"[{config['name']}|seed={seed}] early-stopping at epoch {epoch:03d}; "
                f"best_val_mrr={best_val_mrr:.6f}"
            )
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    if ema is not None:
        # Rebuild EMA state around best model for evaluation consistency.
        ema = EMAHelper(model, decay=config.get('ema_decay', 0.995))

    model.eval()
    with torch.no_grad():
        z = model(train_data.x_dict, train_data.edge_index_dict)
        final_val_metrics = evaluate_embeddings(z['chunk'], z['entity'], val_edges)
        final_test_metrics = evaluate_embeddings(z['chunk'], z['entity'], test_edges)
        val_diag = embedding_diagnostics(z['chunk'], z['entity'], val_edges, positives_map=all_positives_by_chunk)
        test_diag = embedding_diagnostics(z['chunk'], z['entity'], test_edges, positives_map=all_positives_by_chunk)

    return {
        'name': config['name'],
        'phase': config.get('phase', 'unknown'),
        'seed': seed,
        'config': config,
        'model': model,
        'best_epoch': best_epoch,
        'best_val_mrr': best_val_mrr,
        'history': history,
        'val_metrics': final_val_metrics,
        'test_metrics': final_test_metrics,
        'val_diag': val_diag,
        'test_diag': test_diag,
        'residual_weights': model.residual_weights(),
    }


baseline_val_diag = embedding_diagnostics(
    baseline_chunk_emb,
    baseline_entity_emb,
    val_edges,
    positives_map=all_positives_by_chunk,
)
baseline_test_diag = embedding_diagnostics(
    baseline_chunk_emb,
    baseline_entity_emb,
    test_edges,
    positives_map=all_positives_by_chunk,
)

print('Baseline val metrics:', baseline_val_metrics)
print('Baseline test metrics:', baseline_test_metrics)

base_common = {
    'out_dim': 256,
    'max_epochs': MAX_EPOCHS,
    'patience': PATIENCE,
    'lr': 3e-4,
    'weight_decay': 1e-5,
}

phase1_configs = [
    {
        'name': 'ce_r4_h2',
        'phase': 'phase1',
        **base_common,
        'num_layers': 1,
        'dropout': 0.05,
        'num_random_neg': 4,
        'num_hard_neg': 2,
        'loss_name': 'cross_entropy',
        'preserve_weight': 0.02,
        'residual_weight': 0.88,
    },
    {
        'name': 'ce_r6_h4',
        'phase': 'phase1',
        **base_common,
        'num_layers': 1,
        'dropout': 0.05,
        'num_random_neg': 6,
        'num_hard_neg': 4,
        'loss_name': 'cross_entropy',
        'preserve_weight': 0.02,
        'residual_weight': 0.88,
    },
    {
        'name': 'ce_r8_h8',
        'phase': 'phase1',
        **base_common,
        'num_layers': 1,
        'dropout': 0.05,
        'num_random_neg': 8,
        'num_hard_neg': 8,
        'loss_name': 'cross_entropy',
        'preserve_weight': 0.02,
        'residual_weight': 0.88,
    },
    {
        'name': 'rank_r6_h6_m0.10',
        'phase': 'phase1',
        **base_common,
        'num_layers': 1,
        'dropout': 0.05,
        'num_random_neg': 6,
        'num_hard_neg': 6,
        'loss_name': 'pairwise_softplus',
        'margin': 0.10,
        'preserve_weight': 0.03,
        'residual_weight': 0.90,
    },
    {
        'name': 'rank_r8_h8_m0.15',
        'phase': 'phase1',
        **base_common,
        'num_layers': 1,
        'dropout': 0.05,
        'num_random_neg': 8,
        'num_hard_neg': 8,
        'loss_name': 'pairwise_softplus',
        'margin': 0.15,
        'preserve_weight': 0.03,
        'residual_weight': 0.90,
    },
    {
        'name': 'ce_curriculum_neg',
        'phase': 'phase1',
        **base_common,
        'num_layers': 1,
        'dropout': 0.05,
        'num_random_neg': 2,
        'num_hard_neg': 6,
        'curriculum_neg': True,
        'loss_name': 'cross_entropy',
        'preserve_weight': 0.02,
        'residual_weight': 0.88,
    },
]

phase2_configs = [
    {
        'name': 'ce_one_layer_alpha_init_0.75',
        'phase': 'phase2',
        **base_common,
        'num_layers': 1,
        'dropout': 0.05,
        'num_random_neg': 4,
        'num_hard_neg': 2,
        'loss_name': 'cross_entropy',
        'preserve_weight': 0.02,
        'residual_weight': 0.75,
    },
    {
        'name': 'ce_one_layer_alpha_init_0.65',
        'phase': 'phase2',
        **base_common,
        'num_layers': 1,
        'dropout': 0.05,
        'num_random_neg': 4,
        'num_hard_neg': 2,
        'loss_name': 'cross_entropy',
        'preserve_weight': 0.02,
        'residual_weight': 0.65,
    },
    {
        'name': 'ce_one_layer_dropout_0.10',
        'phase': 'phase2',
        **base_common,
        'num_layers': 1,
        'dropout': 0.10,
        'num_random_neg': 4,
        'num_hard_neg': 2,
        'loss_name': 'cross_entropy',
        'preserve_weight': 0.02,
        'residual_weight': 0.88,
    },
    {
        'name': 'ce_two_layer_with_stronger_residual',
        'phase': 'phase2',
        **base_common,
        'num_layers': 2,
        'dropout': 0.10,
        'num_random_neg': 4,
        'num_hard_neg': 2,
        'loss_name': 'cross_entropy',
        'preserve_weight': 0.02,
        'residual_weight': 0.65,
    },
]


def summarize_runs(run_records: list[dict]):
    rows = []
    for r in run_records:
        rows.append({
            'experiment': r['name'],
            'phase': r['phase'],
            'seed': r['seed'],
            'val_MRR': r['val_metrics']['MRR'],
            'test_MRR': r['test_metrics']['MRR'],
            'val_Recall@10': r['val_metrics']['Recall@10'],
            'test_Recall@10': r['test_metrics']['Recall@10'],
            'val_Recall@5': r['val_metrics']['Recall@5'],
            'test_Recall@5': r['test_metrics']['Recall@5'],
            'val_Recall@20': r['val_metrics']['Recall@20'],
            'test_Recall@20': r['test_metrics']['Recall@20'],
            'best_epoch': r['best_epoch'],
            'chunk_alpha': r['residual_weights']['chunk'],
            'entity_alpha': r['residual_weights']['entity'],
            'val_pos_mean': r['val_diag'].get('pos_mean', np.nan),
            'val_neg_mean': r['val_diag'].get('neg_mean', np.nan),
            'val_margin_mean': r['val_diag'].get('margin_mean', np.nan),
            'test_pos_mean': r['test_diag'].get('pos_mean', np.nan),
            'test_neg_mean': r['test_diag'].get('neg_mean', np.nan),
            'test_margin_mean': r['test_diag'].get('margin_mean', np.nan),
        })
    per_seed_df = pd.DataFrame(rows)

    agg_df = per_seed_df.groupby(['phase', 'experiment'], as_index=False).agg(
        val_MRR_mean=('val_MRR', 'mean'),
        val_MRR_std=('val_MRR', 'std'),
        test_MRR_mean=('test_MRR', 'mean'),
        test_MRR_std=('test_MRR', 'std'),
        val_R10_mean=('val_Recall@10', 'mean'),
        val_R10_std=('val_Recall@10', 'std'),
        test_R10_mean=('test_Recall@10', 'mean'),
        test_R10_std=('test_Recall@10', 'std'),
        val_R5_mean=('val_Recall@5', 'mean'),
        test_R5_mean=('test_Recall@5', 'mean'),
        val_R20_mean=('val_Recall@20', 'mean'),
        test_R20_mean=('test_Recall@20', 'mean'),
        best_epoch_mean=('best_epoch', 'mean'),
        chunk_alpha_mean=('chunk_alpha', 'mean'),
        entity_alpha_mean=('entity_alpha', 'mean'),
        val_margin_mean=('val_margin_mean', 'mean'),
        test_margin_mean=('test_margin_mean', 'mean'),
    )

    return per_seed_df, agg_df.sort_values('val_MRR_mean', ascending=False).reset_index(drop=True)


def run_phase(configs: list[dict], control_val_mrr: float | None = None):
    records = []
    for cfg in configs:
        print(f"\n=== {cfg['phase']} :: {cfg['name']} ===")
        for seed in SEEDS:
            records.append(train_experiment_seeded(cfg, seed=seed, control_val_mrr=control_val_mrr))
    return records


phase1_records = run_phase(phase1_configs, control_val_mrr=None)
phase1_per_seed_df, phase1_agg_df = summarize_runs(phase1_records)

control_row = phase1_agg_df[phase1_agg_df['experiment'] == 'ce_r4_h2'].iloc[0]
control_val_mrr = float(control_row['val_MRR_mean'])
control_val_r10 = float(control_row['val_R10_mean'])
print('\nControl (phase1) mean val MRR:', control_val_mrr)

phase1_eligible = phase1_agg_df[
    (phase1_agg_df['val_MRR_mean'] >= control_val_mrr + VAL_IMPROVEMENT_THRESHOLD)
    & (phase1_agg_df['val_R10_mean'] >= control_val_r10)
]
if len(phase1_eligible) == 0:
    # Fallback: keep top-2 by val MRR if none reaches threshold.
    phase1_eligible = phase1_agg_df.sort_values('val_MRR_mean', ascending=False).head(2)

print('\nPhase1 aggregate (sorted):')
print(phase1_agg_df[['experiment', 'val_MRR_mean', 'val_MRR_std', 'test_MRR_mean', 'val_R10_mean', 'test_R10_mean']])
print('\nPhase1 eligible:')
print(phase1_eligible[['experiment', 'val_MRR_mean', 'val_R10_mean']])

phase2_records = run_phase(phase2_configs, control_val_mrr=control_val_mrr)
phase2_per_seed_df, phase2_agg_df = summarize_runs(phase2_records)

print('\nPhase2 aggregate (sorted):')
print(phase2_agg_df[['experiment', 'val_MRR_mean', 'val_MRR_std', 'test_MRR_mean', 'val_R10_mean', 'test_R10_mean']])

phase12_records = phase1_records + phase2_records
phase12_per_seed_df, phase12_agg_df = summarize_runs(phase12_records)

# Select best candidate from phase1+2 by mean val MRR.
phase12_best = phase12_agg_df.sort_values('val_MRR_mean', ascending=False).iloc[0]
best_name_phase12 = phase12_best['experiment']
best_cfg_phase12 = None
for cfg in phase1_configs + phase2_configs:
    if cfg['name'] == best_name_phase12:
        best_cfg_phase12 = cfg
        break

phase3_cfgs = [
    {
        **best_cfg_phase12,
        'name': 'best_from_phase1_or2_plus_cosine_clip',
        'phase': 'phase3',
        'scheduler': 'cosine',
        'grad_clip': 1.0,
        'min_lr': 1e-5,
    },
    {
        **best_cfg_phase12,
        'name': 'best_from_11_plus_ema',
        'phase': 'phase3',
        'scheduler': 'cosine',
        'grad_clip': 1.0,
        'min_lr': 1e-5,
        'use_ema': True,
        'ema_decay': 0.995,
    },
]

phase3_records = run_phase(phase3_cfgs, control_val_mrr=control_val_mrr)
phase3_per_seed_df, phase3_agg_df = summarize_runs(phase3_records)

print('\nPhase3 aggregate (sorted):')
print(phase3_agg_df[['experiment', 'val_MRR_mean', 'val_MRR_std', 'test_MRR_mean', 'val_R10_mean', 'test_R10_mean']])

all_records = phase12_records + phase3_records
all_per_seed_df, all_agg_df = summarize_runs(all_records)

# Final selection rules.
candidate_df = all_agg_df.copy()
candidate_df['val_MRR_std'] = candidate_df['val_MRR_std'].fillna(0.0)
std_filtered = candidate_df[candidate_df['val_MRR_std'] <= STD_THRESHOLD]
if len(std_filtered) == 0:
    std_filtered = candidate_df

std_filtered = std_filtered.sort_values('val_MRR_mean', ascending=False).reset_index(drop=True)
final_row = std_filtered.iloc[0]

# If very close candidates, pick simpler one.
if len(std_filtered) > 1:
    second_row = std_filtered.iloc[1]
    if (final_row['val_MRR_mean'] - second_row['val_MRR_mean']) < 0.003:
        complexity = {
            'ce_r4_h2': 1,
            'ce_r6_h4': 2,
            'ce_r8_h8': 3,
            'rank_r6_h6_m0.10': 3,
            'rank_r8_h8_m0.15': 4,
            'ce_curriculum_neg': 3,
            'ce_one_layer_alpha_init_0.75': 1,
            'ce_one_layer_alpha_init_0.65': 1,
            'ce_one_layer_dropout_0.10': 1,
            'ce_two_layer_with_stronger_residual': 2,
            'best_from_phase1_or2_plus_cosine_clip': 2,
            'best_from_11_plus_ema': 3,
        }
        c1 = complexity.get(final_row['experiment'], 9)
        c2 = complexity.get(second_row['experiment'], 9)
        if c2 < c1:
            final_row = second_row

# Final test guardrail vs control.
control_test_mrr = float(control_row['test_MRR_mean'])
if float(final_row['test_MRR_mean']) < (control_test_mrr - MAX_ALLOWED_TEST_DROP):
    print('\nFinal guardrail triggered: selected candidate underperforms control on test. Reverting to control.')
    final_row = control_row

selected_name = final_row['experiment']
selected_runs = [r for r in all_records if r['name'] == selected_name]
selected_run = max(selected_runs, key=lambda r: r['val_metrics']['MRR'])

selected_experiment = selected_run
selected_model_version = f"graphsage_{selected_name}_multi_seed"

final_val_metrics = selected_run['val_metrics']
test_metrics = selected_run['test_metrics']
experiment_results = all_records

summary_df = all_agg_df.sort_values(['val_MRR_mean', 'test_MRR_mean'], ascending=False).reset_index(drop=True)
per_seed_results_df = all_per_seed_df.sort_values(['experiment', 'seed']).reset_index(drop=True)

comparison_rows = []
for _, row in summary_df.iterrows():
    metrics_val = {
        'MRR': row['val_MRR_mean'],
        'Recall@5': row['val_R5_mean'],
        'Recall@10': row['val_R10_mean'],
        'Recall@20': row['val_R20_mean'],
    }
    metrics_test = {
        'MRR': row['test_MRR_mean'],
        'Recall@5': row['test_R5_mean'],
        'Recall@10': row['test_R10_mean'],
        'Recall@20': row['test_R20_mean'],
    }
    comparison_rows.extend(build_metric_table('val', row['experiment'], metrics_val, baseline_val_metrics))
    comparison_rows.extend(build_metric_table('test', row['experiment'], metrics_test, baseline_test_metrics))
comparison_df = pd.DataFrame(comparison_rows)

diagnostics_rows = [
    {'experiment': 'baseline', 'split': 'val', **baseline_val_diag},
    {'experiment': 'baseline', 'split': 'test', **baseline_test_diag},
]
for r in all_records:
    diagnostics_rows.append({'experiment': r['name'], 'split': 'val', **r['val_diag'], 'seed': r['seed']})
    diagnostics_rows.append({'experiment': r['name'], 'split': 'test', **r['test_diag'], 'seed': r['seed']})
diagnostics_df = pd.DataFrame(diagnostics_rows)

print('\n=== FINAL SELECTION ===')
print('Selected experiment:', selected_name)
print('Selected seed:', selected_run['seed'])
print('Selected val metrics:', final_val_metrics)
print('Selected test metrics:', test_metrics)
print('Residual weights:', selected_run['residual_weights'])
print('\nAggregate leaderboard (top 12):')
print(summary_df[['phase', 'experiment', 'val_MRR_mean', 'val_MRR_std', 'test_MRR_mean', 'test_MRR_std', 'val_R10_mean', 'test_R10_mean']].head(12))

summary_df

In [ ]:
# In this notebook we stop at 3-phase training and comparison.
# Upsert/inference is done in `gnn_retrieval_neo4j.ipynb`.


In [ ]:
# Demo retrieval intentionally omitted in phase notebook.


In [ ]:
driver.close()
